# Imputation quality across datasets

Run this **after** `merge_splits.ipynb` has been run for every dataset - it only
reads what Step 4 wrote and computes nothing about imputation itself.

Each line is one dataset, x is the percentage of nulls injected, y is the metric.
One panel per (strategy, mechanism): averaging over either would hide the effect
being measured, since the mechanism decides *which* cells go missing and the
strategy decides *how* they are filled. Where several seeds exist, the line is
their mean and the wash around it their min-max spread.

Colour comes from the dataset's position in a fixed order, so changing `DATASETS`
never repaints the datasets that remain, and each dataset also carries its own
marker - colour alone is not an identity channel. Every figure is followed by its
table view, so no value is reachable only by eye.

In [ ]:
# ===================== ALL PARAMETERS - CHANGE THEM ONLY HERE =====================
PROJECT_ROOT = None       # None -> auto-detect by walking up from the working directory
DATASETS     = None       # None -> every dataset found; ["DBLPACM", "Beer"] to pick
SEEDS        = None       # None -> every seed found;    [1576] to pin one
DARK         = False      # dark mode is its own validated palette, not a flipped light one
SIM_METRIC   = "avg_sim_imputed"          # cosine(imputed value, truth), over K filled cells
EXACT_METRIC = "exact_pct_of_nulls"       # exact matches as a share of every injected null
SAVE_DIR     = "plots"    # under Data_preparation/; None -> display only, write nothing
DPI          = 200
# ==================================================================================

import sys
from pathlib import Path

_hint = PROJECT_ROOT
_cands = ([Path(_hint).expanduser().resolve()] if _hint
          else [Path.cwd().resolve(), *Path.cwd().resolve().parents])
_roots = [p for p in _cands if (p / "Data_preparation" / "code" / "plot_eval.py").is_file()]
assert _roots, (f"project root not found from {Path.cwd()}. Set PROJECT_ROOT above to the "
                f"SolvIER2 folder, e.g. PROJECT_ROOT = \"/home/u59352/SolvIER2\"")
ROOT = _roots[0]
sys.path.insert(0, str(ROOT / "Data_preparation" / "code"))
from plot_eval import *  # noqa: E402  - load_evaluations, metric_figure, table_view, describe


OUT_DIR = (ROOT / "Data_preparation" / SAVE_DIR) if SAVE_DIR else None
DF = load_evaluations(ROOT, datasets=DATASETS, seeds=SEEDS)
describe(DF)

print("\nsemantic-similarity metrics available:")
for name, what in SIM_METRICS.items():
    print(f"   {name:<28} {what}")
print("exact-match metrics available:")
for name, what in EXACT_METRICS.items():
    print(f"   {name:<28} {what}")

## Semantic similarity

`avg_semantic_sim` is the cosine between the **imputed value** and the
**ground-truth value**, averaged over the cells that were scored - 1.0 is an exact
hit, ~0 unrelated. The variants differ only in the denominator: `_scorable` spreads
the same cosines over every recoverable null and `_all_nulls` over every null,
both charging 0 for a cell the imputer left empty.

The two `sim_x_coverage` metrics scale the similarity by how much of the table was
actually filled, so a cautious strategy cannot score well by filling almost nothing:

    sim_x_imputed_over_nulls = avg_semantic_sim x (imputed / nulls)
    sim_x_imputed_over_cells = avg_semantic_sim x (imputed / (M x N))

`M` is the entity rows in the file and `N` the attributes imputed, so `M x N` is
every attribute cell in the table, null or not. Dividing by it puts the metric on
the scale of the whole table rather than of the damage, which makes the 5% and 40%
null levels directly comparable - the `/nulls` version renormalises at each level
and hides that there was four times as much to repair. A strategy that fills only when
confident scores high on the first and low on the last; that gap is the point.

In [ ]:
# --- this cell can be run on its own: recover what the first cell set up ---
if "DF" not in globals():
    import sys
    from pathlib import Path
    _code = next((p / "Data_preparation" / "code" for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                  if (p / "Data_preparation" / "code" / "plot_eval.py").is_file()), None)
    assert _code, "run the first cell, or start Jupyter inside the SolvIER2 checkout"
    sys.path.insert(0, str(_code))
    from plot_eval import *
    DF = load_evaluations()
    SIM_METRIC, EXACT_METRIC = "avg_semantic_sim", "exact%_of_nulls"
    DARK, SAVE_DIR, DPI = False, None, 200
    print("reloaded", len(DF), "evaluation rows")

fig = metric_figure(DF, SIM_METRIC, dark=DARK, dpi=DPI,
                    ylabel=SIM_METRIC,
                    save_to=OUT_DIR / f"{SIM_METRIC}.png" if OUT_DIR else None)
table_view(DF, SIM_METRIC)

## Exact matches

`exact%_of_nulls` counts the cells the imputer reproduced exactly, as a share of
every injected null - cells it refused to fill included, so refusing is not
rewarded. Use `exact%_of_imputed` to score only what it did fill, or
`exact%_of_scorable` to drop the cells whose ground truth is itself empty. The raw
`exact_matches` count is not comparable across datasets of different sizes.

In [ ]:
# --- this cell can be run on its own: recover what the first cell set up ---
if "DF" not in globals():
    import sys
    from pathlib import Path
    _code = next((p / "Data_preparation" / "code" for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                  if (p / "Data_preparation" / "code" / "plot_eval.py").is_file()), None)
    assert _code, "run the first cell, or start Jupyter inside the SolvIER2 checkout"
    sys.path.insert(0, str(_code))
    from plot_eval import *
    DF = load_evaluations()
    SIM_METRIC, EXACT_METRIC = "avg_semantic_sim", "exact%_of_nulls"
    DARK, SAVE_DIR, DPI = False, None, 200
    print("reloaded", len(DF), "evaluation rows")

fig = metric_figure(DF, EXACT_METRIC, dark=DARK, dpi=DPI,
                    ylabel=EXACT_METRIC,
                    save_to=OUT_DIR / f"{EXACT_METRIC.replace('%', 'pct')}.png" if OUT_DIR else None)
table_view(DF, EXACT_METRIC, decimals=1)

## Write every variant

One PNG per metric, for the paper. Skipped when `SAVE_DIR` is None.

In [ ]:
# --- this cell can be run on its own: recover what the first cell set up ---
if "DF" not in globals():
    import sys
    from pathlib import Path
    _code = next((p / "Data_preparation" / "code" for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                  if (p / "Data_preparation" / "code" / "plot_eval.py").is_file()), None)
    assert _code, "run the first cell, or start Jupyter inside the SolvIER2 checkout"
    sys.path.insert(0, str(_code))
    from plot_eval import *
    DF = load_evaluations()
    SIM_METRIC, EXACT_METRIC = "avg_semantic_sim", "exact%_of_nulls"
    DARK, SAVE_DIR, DPI = False, None, 200
    print("reloaded", len(DF), "evaluation rows")

import matplotlib.pyplot as plt

if OUT_DIR:
    for metric, what in {**SIM_METRICS, **EXACT_METRICS}.items():
        if metric not in DF.columns:
            continue
        fig = metric_figure(DF, metric, dark=DARK, dpi=DPI, ylabel=metric,
                            save_to=OUT_DIR / f"{metric.replace('%', 'pct')}.png")
        plt.close(fig)
    print(f"\nall figures -> {OUT_DIR}")
else:
    print("SAVE_DIR is None - nothing written")